### Librerias

In [2]:
import kfp
from google.cloud import aiplatform
from kfp.v2 import dsl, compiler
from kfp.v2.dsl import component, Input, Output, Dataset
from typing import NamedTuple

import os

### Componente de Lectura y Procesamiento

In [ ]:
@component(
    packages_to_install=[
        "google-cloud-bigquery",
        "google-cloud-bigquery-storage",
        "pandas", "db-dtypes", "pyarrow"]
)
def process_data(
                 project: str, 
                 source_x_train_table: str, 
                 dataset: Output[Dataset]):
    from google.cloud import bigquery
    import pandas as pd
    import pyarrow.parquet as pq

    client = bigquery.Client(project = project)

    Datos = client.query(
        '''SELECT * FROM `{dsource_table}`
        '''.format(dsource_table = source_x_train_table)
    ).to_dataframe()
    DatosDDDDDD
    print('================= Tabla =======================')
    print(Datos.head(5))
    print('================= Dataset.path =================')
    print('path = ', dataset.path)

    Datos.to_parquet(f'{dataset.path}.parquet', engine='pyarrow', index=False)  # Guardar el Parquet dentro de un Cloud Storage


### Componente de Entrenamiento

In [32]:
@component(
    packages_to_install=[
        "google-cloud-bigquery",
        "google-cloud-bigquery-storage",
        "pandas", "db-dtypes", "pyarrow", "joblib", "scikit-learn", "xgboost"]
)
def training_model(
                 project: str, 
                 inputd: Input[Dataset]):
    from google.cloud import bigquery
    from google.cloud import storage
    import pandas as pd
    import pyarrow.parquet as pq
    import joblib
    import xgboost as xgb
    from sklearn.preprocessing import LabelEncoder 

    client = bigquery.Client(project = project)

    Datos = pd.read_parquet(f'{inputd.path}.parquet')
    Xtrain = Datos[Datos.columns[:-1]]
    Ytrain = Datos['Species']

    print('================= Encoding =======================')
    encoder = LabelEncoder()
    Ytrain2 = encoder.fit_transform(Ytrain)    
    print(pd.concat([Ytrain, pd.DataFrame(Ytrain2)],axis=1).head(5))

    Modelo = xgb.XGBClassifier( n_estimators=100,
                                max_depth=3,
                                learning_rate=0.1,
                                random_state=42 )
    Modelo.fit(Xtrain, Ytrain2)

    # serialization into a binary file (writes file to disk)
    #-----MODELO
    modelName = "xgb_model.joblib"
    joblib.dump(Modelo, modelName)
    #-----ENCODER
    encoderName = "label_encoder.joblib"
    joblib.dump(encoder, encoderName)

    # Llevamos a un Bucket de Cloud Storage
    #-----MODELO
    storage_client = storage.Client(project=project)
    bucket_name = "cloudstorage-mlops"
    destination_blob_name = "Training-Pipeline/Data/Model/XGBmodel.joblib"
    
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(modelName)

    #-----ENCODER
    destination_blob_name = "Training-Pipeline/Data/Model/labelEncoder.joblib"   

    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(encoderName) 

### Pipeline Principal

In [ ]:
@kfp.dsl.pipeline(
    name="training-pipeline",
    description="intro",
    pipeline_root="gs://cloudstorage-mlops/Training-Pipeline"
)

def main_pipeline(
    source_x_train_table: str, 
    features_table: str,
    project: str = "proyecto-mlops-504619",
    gcp_region: str = "us-central1",    
):
    #--------- 1er Componente ---------#
    get_data = process_data(


        ddsfddd
        project = project,
        source_x_train_table = source_x_train_table )
    
    get_data.set_display_name("PROCESS_DATA")

    #--------- 2do Componente ---------#
    train = training_model(
        project = project,
        inputd = get_data.output ).after(get_data)

    train.set_display_name("TRAINING_PROCESS")

### Compilador

In [ ]:
from kfp.v2 import compiler as v2_compiler

v2_compiler.Compiler().compile(
    pipeline_func=main_pipeline,
    package_path="TrainingPipeline-Proyecto.json"
)

/home/guy3hil/vertex_dev/lib/python3.8/site-packages/kfp/v2/compiler/compiler.py:1290: FutureWarning: APIs imported from the v1 namespace (e.g. kfp.dsl, kfp.components, etc) will not be supported by the v2 compiler since v2.0.0
  warnings.warn(


### Job en Vertex

In [ ]:
from google.cloud import aiplatform
aiplatform.init(project="proyecto-mlops-504619", location="us-central1")

job = aiplatform.PipelineJob(
    display_name="Pipeline de Entrenamiento",
    template_path="TrainingPipeline-Proyecto.json",
    enable_caching=False,
    project="proyecto-mlops-504619",
    location="us-central1",
    parameter_values={"source_x_train_table":"proyecto-fuente.BaseDatosIris.iris-train"}
)

print('submit pipeline job ...')
job.submit(service_account="vertex-processing@proyecto-mlops-504619.iam.gserviceaccount.com")

submit pipeline job ...
Creating PipelineJob
PipelineJob created. Resource name: projects/884292398314/locations/us-central1/pipelineJobs/training-pipeline-20260817032613
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/884292398314/locations/us-central1/pipelineJobs/training-pipeline-20260817032613')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/training-pipeline-20260817032613?project=884292398314
